# Stage 9: the old watch model vs the new one

**Old:** the still/moving forest on the watch today (`firmware/FinalPrototype/har_model.h`): 10 trees, 5 numbers from the **size** of the acceleration over the last **2.56 s**, a new answer every **0.5 s**. It learned from UCI-HAR (a phone on the waist, young adults, a lab).

**New:** Stage 8's XGBoost + **live** HMM: 16 features from **30 s**, 4 classes, trained on Capture-24.

Both are run here **exactly as on the watch**: the old one through the firmware's own C files (compiled on this laptop, not changed), the new one with the live HMM that only uses the past.

1. Capture-24's 30 test people (the new model here is the **121-person tested twin**, because the final model has seen these people)
2. GOTOV's 35 older adults (seen by neither), with GOTOV's axes lined up as in Stage 6
3. Your own watch recordings: `test_watch_recording.py` does the same comparison on them

**Kernel:** `.venv-tf` in this folder. Prediction only, no training.

## How this works (for beginners)

**Different questions, so compare on the shared one.** The old model only says *still* or *moving*. The new one says sleep / sedentary / light / moderate-vigorous, which turns into still (sleep, sedentary) or moving (light, moderate-vigorous). So the fair comparison is **still vs moving**.

**Different time steps.** The old model answers every 0.5 s; the new one every 30 s. For each 30 s window, the old model's ~55 answers are combined by majority: moving if more than half say moving.

**Two rules for "moving":**

| Rule | Standing counts as | Who it suits |
|---|---|---|
| Capture-24's | moving (light) | the new model learned this |
| Physical | still | the old model learned this (UCI) |

Capture-24 only has its own rule. GOTOV says exactly what each person was doing, so both rules can be scored there.

In [1]:
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

from common import CLASSES, DATA_DIR, FEATURES, RESULTS_DIR, paired_bootstrap, scores, split_people, to_binary
from test_watch_recording import FIRMWARE, OLD_HARNESS, live_hmm

WORK = Path('processed/old_model_c'); WORK.mkdir(exist_ok=True)
(WORK / 'old_model_harness.c').write_text(OLD_HARNESS)
exe = WORK / 'old_model_harness.exe'
build = subprocess.run([sys.executable, '-m', 'ziglang', 'cc', '-O2', '-w', f'-I{FIRMWARE}', '-o', str(exe), str(WORK / 'old_model_harness.c')],
                       capture_output=True, text=True)
assert build.returncode == 0, build.stderr
print('old model compiled from', FIRMWARE / 'har_model.h')

def old_model_on_mag(mag2d):
    """The firmware's still/moving forest on each row of magnitudes (each row starts a fresh 2.56 s buffer, as after
    a gap). -> share of its answers that say moving, per row."""
    mag = mag2d.astype(np.float32).ravel()
    gap = np.zeros(len(mag), dtype=np.int32); gap[::mag2d.shape[1]] = 1
    with open(WORK / 'in.bin', 'wb') as fh:
        np.array([len(mag)], dtype=np.int32).tofile(fh)
        np.rec.fromarrays([mag, gap], names='m,g').tofile(fh)
    subprocess.run([str(exe), str(WORK / 'in.bin'), str(WORK / 'out.bin')], check=True)
    d = np.fromfile(WORK / 'out.bin', dtype=np.int32).reshape(mag2d.shape).astype(float)
    d[d < 0] = np.nan
    return np.nanmean(d, axis=1)

def old_model_on_windows(X, scale=1.0):
    return old_model_on_mag(scale * np.linalg.norm(X, axis=2))

# check: the firmware's own boot self-test windows must give the answers the firmware expects
import re
tv = (FIRMWARE / 'test_vectors.h').read_text()
tv_mag = np.array([float(v.strip().rstrip('f')) for v in re.search(r'TV_HAR_MAG\[\d+\] = \{([^}]*)\}', tv).group(1).split(',') if v.strip()])
tv_class = [int(v) for v in re.search(r'TV_HAR_CLASS\[TV_N\] = \{([^}]*)\}', tv).group(1).split(',')]
got = old_model_on_mag(tv_mag.reshape(len(tv_class), -1))
assert list(got.astype(int)) == tv_class, (got, tv_class)
print('old model reproduces the firmware self-test answers:', tv_class)

old model compiled from C:\Users\syafi\SilverGuard Major Project\firmware\FinalPrototype\har_model.h
old model reproduces the firmware self-test answers: [1, 1, 0]


## 1. Capture-24's 30 test people

In [2]:
train_ids, test_ids = split_people()
twin = xgb.XGBClassifier(); twin.load_model(RESULTS_DIR / 'xgb_watch.json')
hmm = np.load(RESULTS_DIR / 'hmm.npz')
f = np.load(DATA_DIR / 'features_all.npz')
rows = []
for p in test_ids:
    with np.load(DATA_DIR / f'{p}.npz') as g:
        X, y = g['X'], g['y4'].astype(int)
    F = f['F'][f['pid'] == p]
    new = live_hmm(twin.predict_proba(F), hmm['start'], hmm['transition'])
    old_share = old_model_on_windows(X)
    rows.append(pd.DataFrame({'pid': p, 'y': y, 'new': new, 'old_share': old_share}))
c24 = pd.concat(rows, ignore_index=True)
c24['old_moving'] = (c24.old_share > 0.5).astype(int)
c24.to_csv(RESULTS_DIR / 'stage9_capture24_windows.csv', index=False)

t = to_binary(c24.y)
res = pd.DataFrame({'old (watch today)': scores(t, c24.old_moving, 2), 'new (XGBoost + live HMM)': scores(t, to_binary(c24.new), 2)}).T
print(f'{len(c24):,} windows, still vs moving by Capture-24\'s rule:')
print(res.round(3).to_string())
per_class = c24.groupby(c24.y.map(dict(enumerate(CLASSES)))).agg(
    windows=('y', 'size'), old_moving_pct=('old_moving', lambda v: 100 * v.mean()),
    new_moving_pct=('new', lambda v: 100 * (v >= 2).mean())).reindex(CLASSES)
print('\n% of each true class called moving:')
print(per_class.round(0).to_string())
per = c24.groupby('pid').apply(lambda d: pd.Series({'old': scores(to_binary(d.y), d.old_moving, 2)['kappa'],
                                                     'new': scores(to_binary(d.y), to_binary(d.new), 2)['kappa']}), include_groups=False)
m, lo, hi = paired_bootstrap(per.new, per.old)
print(f'\nperson by person: new better for {(per.new > per.old).sum()} of {len(per)}; '
      f'mean kappa difference {m:+.3f} (95% range {lo:+.3f} to {hi:+.3f})')

58,152 windows, still vs moving by Capture-24's rule:
                          accuracy  macro-F1  kappa
old (watch today)            0.887     0.813  0.626
new (XGBoost + live HMM)     0.918     0.869  0.739

% of each true class called moving:
                   windows  old_moving_pct  new_moving_pct
y                                                         
sleep                22947             1.0             1.0
sedentary            24075            11.0            10.0
light                 8439            60.0            75.0
moderate-vigorous     2691            90.0            96.0



person by person: new better for 28 of 30; mean kappa difference +0.118 (95% range +0.086 to +0.150)


## 2. GOTOV: 35 older adults, seen by neither model

GOTOV names each activity, so both rules for "moving" can be scored. The new model reads GOTOV's wrist with its axes lined up (Stage 6). The old model only uses the size of the acceleration, so the axes don't matter to it.

In [3]:
s5 = np.load(RESULTS_DIR / 'stage5_gotov_predictions.npz')
Fg = np.load('processed/gotov/features_rotated.npz')['F']
Xg = np.concatenate([np.load(f'processed/gotov/{p}.npz')['X'][np.argsort(np.load(f'processed/gotov/{p}.npz')['start'])]
                     for p in sorted(np.unique(s5['pid']))])
assert len(Xg) == len(s5['y'])
new_g = np.zeros(len(Fg), dtype=int)
pr = twin.predict_proba(Fg)
for p in np.unique(s5['pid']):
    m_ = s5['pid'] == p
    new_g[m_] = live_hmm(pr[m_], hmm['start'], hmm['transition'])
old_g = old_model_on_windows(Xg)
act = s5['activity']
physical_still = np.isin(act, ['lyingDownLeft', 'lyingDownRight', 'sittingSofa', 'sittingCouch', 'sittingChair', 'standing'])
gv = pd.DataFrame({'pid': s5['pid'], 'activity': act, 'y': s5['y'], 'new': new_g, 'old_share': old_g,
                   'physical_moving': (~physical_still).astype(int)})
gv['old_moving'] = (gv.old_share > 0.5).astype(int)
gv.to_csv(RESULTS_DIR / 'stage9_gotov_windows.csv', index=False)

res_g = {}
for rule, truth in [("Capture-24's rule (standing = moving)", to_binary(gv.y)), ('physical rule (standing = still)', gv.physical_moving)]:
    res_g[f'old, {rule}'] = scores(truth, gv.old_moving, 2)
    res_g[f'new, {rule}'] = scores(truth, to_binary(gv.new), 2)
res_g = pd.DataFrame(res_g).T
print(res_g.round(3).to_string())
by_act = gv.groupby('activity').agg(windows=('y', 'size'), old_moving_pct=('old_moving', lambda v: 100 * v.mean()),
                                    new_moving_pct=('new', lambda v: 100 * (v >= 2).mean()))
order = ['lyingDownLeft', 'lyingDownRight', 'sittingSofa', 'sittingCouch', 'sittingChair', 'standing', 'dishwashing',
         'stakingShelves', 'vacuumCleaning', 'walkingSlow', 'walkingNormal', 'walkingFast', 'cycling', 'step']
print('\n% of each activity called moving:')
print(by_act.reindex([a for a in order if a in by_act.index]).round(0).to_string())
per_g = {}
for rule, col in [("Capture-24's rule", None), ('physical rule', 'physical_moving')]:
    pp = gv.groupby('pid').apply(lambda d: pd.Series({
        'old': scores(to_binary(d.y) if col is None else d[col], d.old_moving, 2)['kappa'],
        'new': scores(to_binary(d.y) if col is None else d[col], to_binary(d.new), 2)['kappa']}), include_groups=False).dropna()
    m, lo, hi = paired_bootstrap(pp.new, pp.old)
    per_g[rule] = (m, lo, hi, int((pp.new > pp.old).sum()), len(pp))
    print(f'{rule}: new better for {(pp.new > pp.old).sum()} of {len(pp)} people; mean kappa difference {m:+.3f} (95% range {lo:+.3f} to {hi:+.3f})')

                                            accuracy  macro-F1  kappa
old, Capture-24's rule (standing = moving)     0.934     0.931  0.862
new, Capture-24's rule (standing = moving)     0.968     0.966  0.932
old, physical rule (standing = still)          0.979     0.979  0.957
new, physical rule (standing = still)          0.940     0.938  0.876

% of each activity called moving:
                windows  old_moving_pct  new_moving_pct
activity                                               
lyingDownLeft       209             0.0            13.0
lyingDownRight      206             0.0             0.0
sittingSofa         206             1.0             0.0
sittingCouch        210             1.0             0.0
sittingChair        201             1.0             0.0
standing            130             1.0            80.0
dishwashing         209            99.0            83.0
stakingShelves      192            97.0           100.0
vacuumCleaning      190            90.0           100.0

Capture-24's rule: new better for 33 of 35 people; mean kappa difference +0.084 (95% range +0.060 to +0.110)


physical rule: new better for 6 of 35 people; mean kappa difference -0.119 (95% range -0.160 to -0.079)


## 3. What this shows

**1. In real daily life (Capture-24), the new model is clearly better.** Still vs moving kappa **0.739 vs 0.626**, better for 28 of 30 people (+0.118, 95% range +0.086 to +0.150). The difference is mostly **light activity**: the new model catches 75% of it, the old one 60%. Both are equally good on sleep and sitting.

**2. On older adults (GOTOV), the answer depends on what "moving" means:**

| Rule | Old | New | Person by person |
|---|---|---|---|
| Capture-24's (standing = moving) | 0.862 | **0.932** | new better for 33 of 35 |
| Physical (standing = still) | **0.957** | 0.876 | **old** better for 29 of 35 |

The new model calls **standing** "moving" 80% of the time, on purpose: Capture-24, which it learned from, counts standing as light activity. The old model learned from UCI, where standing is still, and GOTOV's standing is very calm (a scripted rest), so the old model gets it right almost every time.

**3. Which is "right" is a product decision, not a model question:**
- If SilverGuard wants **"is this person physically active?"** (activity minutes, moderate-vigorous), the new model's rule is the standard one in research, and it is the better model
- If SilverGuard wants **"is this person on their feet or not?"** (for example, an inactivity alert after a fall), standing should count as still, and on GOTOV the old model does that better. The new model would need standing separated from light activity, which its 4 classes don't do

**4. The old model has two real strengths:** it answers every **0.5 s** (the new one every 30 s), and it ignores the axes completely (it only uses the size of the acceleration), so it needs no alignment step.

**5. Caveats:** GOTOV's still moments are much calmer than real life (Stage 5), which flatters a model that just looks at how much the wrist moves. Capture-24 can only be scored by its own rule. And neither model has been tested on the SilverGuard watch yet: `test_watch_recording.py` does this same comparison on your own recordings, scored both ways.

In [4]:
# every claim above, checked against the numbers
r = res['kappa']; rg = res_g['kappa']
assert round(r['new (XGBoost + live HMM)'], 3) == 0.739 and round(r['old (watch today)'], 3) == 0.626
mm, ll, hh = paired_bootstrap(per.new, per.old)
assert (per.new > per.old).sum() == 28 and [round(v, 3) for v in (mm, ll, hh)] == [0.118, 0.086, 0.150]
assert round(per_class.loc['light', 'new_moving_pct']) == 75 and round(per_class.loc['light', 'old_moving_pct']) == 60
cr, pr_ = "old, Capture-24's rule (standing = moving)", 'old, physical rule (standing = still)'
assert [round(rg[k], 3) for k in [cr, cr.replace('old', 'new'), pr_, pr_.replace('old', 'new')]] == [0.862, 0.932, 0.957, 0.876]
assert per_g["Capture-24's rule"][3:] == (33, 35) and per_g['physical rule'][3] == 6
assert round(by_act.loc['standing', 'new_moving_pct']) == 80 and round(by_act.loc['standing', 'old_moving_pct']) == 1
print('every claim above matches the numbers')

every claim above matches the numbers
